# Spaceship Titanic: Shared Baseline Notebook

This notebook guides us through our first complete machine learning pipeline. The goal is to build a simple, understandable baseline before trying more advanced features or models.

**How to use this notebook:** Make your own copy in Colab or work on your own GitHub branch. Run each section in order, read the explanations, and record questions or observations in the spaces provided.

## 1. Project Objective

The Spaceship Titanic competition asks us to predict whether each passenger was transported to another dimension.

- Each row represents one passenger.
- The target column is `Transported`.
- Our first goal is to create a reliable baseline model and understand every step.
- We will evaluate our model using accuracy on a validation set.

**Team note:** A baseline is a simple first model that gives us a point of comparison. It does not need to be our final or highest-scoring model.

## 2. Imports

We import the libraries used for loading, preparing, modeling, and evaluating the data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

pd.set_option("display.max_columns", None)

## 3. Load the Data

Upload the competition CSV files to Colab first, or update the file paths if your files are stored in a different location.

In [ ]:
train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")
sample_submission = pd.read_csv("/content/sample_submission.csv")

print("Training data loaded:", train.shape)
print("Test data loaded:", test.shape)
print("Sample submission loaded:", sample_submission.shape)

## 4. Understand the Data Shape

The training data includes the target column. The test data does not include the answers because those are what we must predict for Kaggle.

In [ ]:
display(train.head())
display(train.tail())
print("Training rows and columns:", train.shape)
print("Test rows and columns:", test.shape)
print("Training columns:", list(train.columns))

## 5. Identify Input Features and Targets

`X` contains the information given to the model. `y` contains the correct answer the model will learn to predict.

For this first baseline, we will temporarily leave out `PassengerId`, `Name`, and `Cabin`. These columns may be useful later, but they require additional feature engineering.

In [ ]:
target = "Transported"
columns_to_drop_for_baseline = [target, "PassengerId", "Name", "Cabin"]

X = train.drop(columns=columns_to_drop_for_baseline)
y = train[target].astype(int)

print("Input features:", list(X.columns))
print("Target values:")
display(y.value_counts())

## 6. Check Missing Values

Many machine learning models cannot work directly with blank values. We will handle them later in the preprocessing pipeline. First, identify where they occur.

In [ ]:
missing_values = train.isna().sum().sort_values(ascending=False)
display(missing_values[missing_values > 0])

## 7. Define the Validation Split

We keep 20% of the labeled training data aside as validation data. The model learns from the training portion, and we use the validation portion to estimate how it performs on new passengers.

`stratify=y` helps keep the same proportion of transported and non-transported passengers in both portions.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_valid.shape[0])

## 8. Create a Simple Baseline

We will use logistic regression as our first model. It is a simple classification model that works well as a starting point.

Before modeling, we need to prepare the columns:

- Numerical columns: fill missing values with the median.
- Categorical columns: fill missing values with the most common value, then one-hot encode them.
- The pipeline applies these steps consistently to training and validation data.

In [ ]:
categorical_features = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP"
]

numerical_features = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train, y_train)
print("Baseline model trained successfully.")

## 9. Evaluate Predictions

We now predict the validation rows and compare those predictions with the known answers.

In [ ]:
validation_predictions = baseline_model.predict(X_valid)
validation_accuracy = accuracy_score(y_valid, validation_predictions)

print(f"Validation accuracy: {validation_accuracy:.4f}")
print()
print(classification_report(y_valid, validation_predictions))

## 10. Autoregressive Forecasting

This section is intentionally not applicable to Spaceship Titanic. Autoregressive forecasting is used for time-series problems, where earlier values help predict later values. Our competition is a standard supervised classification problem, so we do not use autoregressive forecasting here.

For this project, this section is a reminder that we should choose methods based on the type of problem we are solving.

## 11. Create a Submission File

After evaluating the baseline, we can train it on all available labeled training data and predict the test passengers.

In [ ]:
test_features = test.drop(columns=["PassengerId", "Name", "Cabin"])

final_model = baseline_model.fit(X, y)
test_predictions = final_model.predict(test_features).astype(bool)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": test_predictions
})

submission.to_csv("spaceship_baseline_submission.csv", index=False)
display(submission.head())
print("Submission file created: spaceship_baseline_submission.csv")

## 12. Conclusion

Complete the following reflection after running the notebook:

- What validation accuracy did the baseline achieve? - 0.7872, or 78.72%
- What does each preprocessing step do?- The preprocessing steps fills in missing values using a method based on the type of feature they are (categorical or numerical), since it would not be appropriate to use the same method for both types of features. 
- Which column or issue was most confusing? - I had the hardest time understanding the results of step 9 in which we evaluate the baseline model's predictions. I'm unfamiliar  with the significance and concept of recall, f1-score, and support.
- What feature would you like us to investigate next? - Evaluating baseline model predictions.

Our next step will be to compare the baseline with engineered features and another lightweight model.